### 1. Imports

In [23]:
import sys
import pandas as pd
from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid
from pathlib import Path

output_dir = Path('../../results/rf')
sys.path.insert(0, r'D:\03_Python\sakura_analytics\src')

from model_data import (
    load_features,
    feature_columns,
    cv_folds,
    evaluate,
    TARGET,
)

### 2. Create the four datasets

In [24]:
# import datasets
df_feb_large = load_features('feb1', source='csv')
df_mar_large = load_features('mar1', source='csv')
df_mar15_large = load_features('mar15', source='csv')

# filter datasets to only include stations in 2025
stations_2025 = df_mar_large.loc[df_mar_large['bloom_year'] == 2025,'station_id',].unique()
df_feb_small = df_feb_large[df_feb_large['station_id'].isin(stations_2025)].copy()
df_mar_small = df_mar_large[df_mar_large['station_id'].isin(stations_2025)].copy()
df_mar15_small = df_mar15_large[df_mar15_large['station_id'].isin(stations_2025)].copy()

# Execution order
datasets = {
    'Mar-15-large': df_mar15_large,
    'Mar-15-small': df_mar15_small,
    'Mar-large': df_mar_large,
    'Mar-small': df_mar_small,
    'Feb-large': df_feb_large,
    'Feb-small': df_feb_small,
}

# Display dataset summary
dataset_summary = []
for name, data in datasets.items():
    dataset_summary.append({
        'dataset': name,
        'rows': len(data),
        'stations': data['station_id'].nunique(),
        'development_rows': (data['split'] == 'development').sum(),
        'test_rows': (data['split'] == 'test').sum(),
        'features':len(feature_columns(data)),
    })
display(pd.DataFrame(dataset_summary))


,dataset,rows,stations,development_rows,test_rows,features
0,Mar-15-large,4365,82,3837,528,36
1,Mar-15-small,2972,48,2444,528,36
2,Mar-large,4365,82,3837,528,23
3,Mar-small,2972,48,2444,528,23
4,Feb-large,4365,82,3837,528,20
5,Feb-small,2972,48,2444,528,20


### 3. Set the shared parameter grid

In [25]:
param_grid = {
    'n_estimators': [200, 500],
    'max_depth': [5, 10, 15, None],
    'min_samples_leaf': [1, 10, 30, 50],
    'max_features': ['sqrt', 0.7, 1.0],
}

candidates = list(ParameterGrid(param_grid))

print('Candidates per dataset:', len(candidates))
print('Total model fits:', len(candidates) * 5 * len(datasets))

Candidates per dataset: 96
Total model fits: 2880


In [26]:
# Compare model features across forecast dates.
features = {
    'Feb 1': set(feature_columns(df_feb_large)),
    'Mar 1': set(feature_columns(df_mar_large)),
    'Mar 15': set(feature_columns(df_mar15_large)),
}

all_features = sorted(set.union(*features.values()))

feature_comparison = pd.DataFrame(
    {
        date: [feature in columns for feature in all_features]
        for date, columns in features.items()
    },
    index=all_features,
)

feature_comparison.index.name = 'feature'

# Show only features that differ across forecast dates.
feature_diff = feature_comparison[
    feature_comparison.nunique(axis=1) > 1
]

# display(feature_diff.replace({True: 'Yes', False: '-'}))
feature_lists = feature_comparison.replace({True: 'Yes', False: '-'})
# display(feature_lists)
# feature_lists.to_csv('../../results/feature_lists.csv', index=True)

### 4. Run grid search and display results for each dataset

In [27]:
# normal: use all original training rows in each fold.
# 30y: use only the latest 30 calendar years of each fold's training rows.
training_windows = {
    'normal': None,
    '30y': 30,
}

all_results = {}

print('Candidates per condition:', len(candidates))
print('Total conditions:', len(datasets) * len(training_windows))
print(
    'Total model fits:',
    len(candidates) * 5 * len(datasets) * len(training_windows),
    flush=True,
)


for base_name, df in datasets.items():
    features = feature_columns(df)
    original_folds = list(cv_folds(df))

    for window_name, window_years in training_windows.items():
        dataset_name = f'{base_name}-{window_name}'

        folds = []
        period_rows = []

        # Prepare training and validation rows.
        for fold, original_train, valid in original_folds:
            # assert not original_train.empty
            # assert not valid.empty
            # assert (
            #     original_train['bloom_year'].max()
            #     < valid['bloom_year'].min()
            # )
            # assert valid['bloom_year'].max() <= 2014

            train_end = int(original_train['bloom_year'].max())

            if window_years is None:
                train = original_train.copy()
                train_start = int(train['bloom_year'].min())
            else:
                train_start = train_end - window_years + 1

                train = original_train.loc[
                    original_train['bloom_year'].between(
                        train_start,
                        train_end,
                    )
                ].copy()

            # assert not train.empty

            # Keep the original validation rows unchanged.
            folds.append((fold, train, valid))

            period_rows.append({
                'fold': fold,
                'train_start': train_start,
                'train_end': train_end,
                'valid_start': int(valid['bloom_year'].min()),
                'valid_end': int(valid['bloom_year'].max()),
                'train_rows': len(train),
                'valid_rows': len(valid),
            })
        training_periods = pd.DataFrame(period_rows)

        fold_rows = []

        # Search every parameter combination for this condition.
        for candidate_id, params in enumerate(candidates):
            candidate_mae = []

            for fold, train, valid in folds:
                model = RandomForestRegressor(
                    **params,
                    random_state=42,
                    n_jobs=-1,
                )

                model.fit(train[features], train[TARGET],)
                train_scores = evaluate(train[TARGET], model.predict(train[features]))
                valid_scores = evaluate(valid[TARGET], model.predict(valid[features]))

                fold_rows.append({
                    'candidate_id': candidate_id,
                    'fold': fold,
                    'train_MAE': train_scores['MAE'],
                    'valid_MAE': valid_scores['MAE'],
                    'valid_RMSE': valid_scores['RMSE'],
                    'valid_R2': valid_scores['R2'],
                    'valid_hit_rate_3d': valid_scores['hit_rate_3d'],
                })

                candidate_mae.append(valid_scores['MAE'])

        fold_results = pd.DataFrame(fold_rows)

        # Equal-weight average across the five validation folds.
        mean_scores = (
            fold_results
            .groupby('candidate_id')
            .agg(
                train_MAE=('train_MAE', 'mean'),
                valid_MAE=('valid_MAE', 'mean'),
                valid_MAE_std=('valid_MAE', 'std'),
                valid_RMSE=('valid_RMSE', 'mean'),
                valid_R2=('valid_R2', 'mean'),
                valid_hit_rate_3d=('valid_hit_rate_3d', 'mean'),
            )
        )

        parameter_table = pd.DataFrame(candidates)
        parameter_table.index.name = 'candidate_id'

        gs_results = (
            parameter_table
            .join(mean_scores)
            .sort_values('valid_MAE', kind='stable')
        )

        best_id = int(gs_results.index[0])

        # Preserve original parameter types, including max_depth=None.
        best_params = candidates[best_id].copy()

        best_fold_results = (
            fold_results.loc[
                fold_results['candidate_id'] == best_id
            ]
            .sort_values('fold')
            .reset_index(drop=True)
        )

        all_results[dataset_name] = {
            'base_dataset': base_name,
            'training_window': window_name,
            'features': features.copy(),
            'best_id': best_id,
            'best_params': best_params,
            'gs_results': gs_results,
            'fold_results': fold_results,
            'best_fold_results': best_fold_results,
            'training_periods': training_periods,
        }

        # Save this condition immediately after its grid search finishes.
        # Example: Mar15-large-30y -> mar15_large_30y
        file_prefix = dataset_name.lower().replace('-', '_')

        gs_results.to_csv(
            output_dir / f'{file_prefix}_grid_search_all_candidates.csv',
            index=True,
            index_label='candidate_id',
            encoding='utf-8-sig',
        )

        fold_results.to_csv(
            output_dir / f'{file_prefix}_all_candidates_fold_results.csv',
            index=False,
            encoding='utf-8-sig',
        )

        best_fold_results.to_csv(
            output_dir / f'{file_prefix}_best_candidate_fold_results.csv',
            index=False,
            encoding='utf-8-sig',
        )

        # Update the comparison of all completed best models.
        comparison_rows = []

        for completed_name, result in all_results.items():
            best = result['gs_results'].loc[result['best_id']]

            comparison_rows.append({
                'dataset': completed_name,
                'base_dataset': result['base_dataset'],
                'training_window': result['training_window'],
                'features': len(result['features']),
                'best_candidate_id': result['best_id'],
                **result['best_params'],
                'train_MAE': best['train_MAE'],
                'valid_MAE': best['valid_MAE'],
                'valid_MAE_std': best['valid_MAE_std'],
                'valid_RMSE': best['valid_RMSE'],
                'valid_R2': best['valid_R2'],
                'valid_hit_rate_3d': best['valid_hit_rate_3d'],
            })

        comparison = pd.DataFrame(comparison_rows).set_index('dataset')

        comparison.to_csv(
            output_dir / 'best_models_comparison.csv',
            index=True,
            encoding='utf-8-sig',
        )

        print(
            f'Saved: {file_prefix} '
            f'({len(all_results)}/12 conditions completed)',
            flush=True,
        )

        print('Best parameters:', best_params)

        print('Top 10 candidates:\n')
        display(gs_results.head(10))

        print('Fold results for the best candidate:\n')
        display(best_fold_results)

display(comparison.round(3))

Candidates per condition: 96
Total conditions: 12
Total model fits: 5760
Saved: mar_15_large_normal (1/12 conditions completed)
Best parameters: {'max_depth': 15, 'max_features': 1.0, 'min_samples_leaf': 10, 'n_estimators': 200}
Top 10 candidates:



,max_depth,max_features,min_samples_leaf,n_estimators,train_MAE,valid_MAE,valid_MAE_std,valid_RMSE,valid_R2,valid_hit_rate_3d
candidate_id,,,,,,,,,,
66,15.0,1.0,10,200,1.930028,3.135788,0.987062,3.895019,0.865916,0.574323
90,NaN,1.0,10,200,1.929310,3.135856,0.987593,3.894611,0.865949,0.574323
42,10.0,1.0,10,200,2.017178,3.139740,1.010844,3.899935,0.865695,0.577518
67,15.0,1.0,10,500,1.929453,3.140489,1.004886,3.884457,0.866774,0.586964
91,NaN,1.0,10,500,1.928689,3.140682,1.005227,3.884468,0.866777,0.586964
43,10.0,1.0,10,500,2.016606,3.143048,1.024966,3.888909,0.866579,0.578631
34,10.0,0.7,10,200,2.037350,3.145636,1.046135,3.901201,0.865720,0.582289
58,15.0,0.7,10,200,1.958054,3.155860,1.010197,3.906010,0.865346,0.583106
82,NaN,0.7,10,200,1.957644,3.155917,1.011826,3.909110,0.865162,0.582597


Fold results for the best candidate:



,candidate_id,fold,train_MAE,valid_MAE,valid_RMSE,valid_R2,valid_hit_rate_3d
0,66,1,1.940313,2.225836,2.901263,0.937816,0.716216
1,66,2,1.922363,3.162658,4.202519,0.795078,0.590551
2,66,3,1.933766,2.732660,3.648083,0.890649,0.650943
3,66,4,1.928652,2.759397,3.240311,0.889909,0.632653
4,66,5,1.925046,4.798391,5.482922,0.816126,0.281250


Saved: mar_15_large_30y (2/12 conditions completed)
Best parameters: {'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 500}
Top 10 candidates:



...

Saved: mar_small_normal (7/12 conditions completed)
Best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200}
Top 10 candidates:



,max_depth,max_features,min_samples_leaf,n_estimators,train_MAE,valid_MAE,valid_MAE_std,valid_RMSE,valid_R2,valid_hit_rate_3d
candidate_id,,,,,,,,,,
24,10.0,sqrt,1,200,2.010090,3.879883,1.861308,4.670781,0.795525,0.481250
35,10.0,0.7,10,500,2.469662,3.890471,1.827343,4.636259,0.797409,0.464583
93,NaN,1.0,30,500,2.895727,3.890564,1.812118,4.654740,0.798500,0.481250
69,15.0,1.0,30,500,2.895727,3.890564,1.812118,4.654740,0.798500,0.481250
45,10.0,1.0,30,500,2.897461,3.890811,1.812295,4.655077,0.798476,0.483333
34,10.0,0.7,10,200,2.471567,3.891080,1.824412,4.642343,0.796953,0.470833
27,10.0,sqrt,10,500,2.665774,3.895442,1.800701,4.669943,0.796943,0.470833
75,NaN,sqrt,10,500,2.582940,3.899396,1.797635,4.670981,0.796447,0.468750
26,10.0,sqrt,10,200,2.669234,3.899470,1.807779,4.680387,0.796092,0.472917


Fold results for the best candidate:



,candidate_id,fold,train_MAE,valid_MAE,valid_RMSE,valid_R2,valid_hit_rate_3d
0,24,1,1.974429,1.964105,2.801494,0.939005,0.770833
1,24,2,1.968868,4.622883,5.729122,0.622235,0.406250
2,24,3,2.033589,2.793371,3.623028,0.898376,0.656250
3,24,4,2.032880,3.291546,3.936298,0.840740,0.510417
4,24,5,2.040684,6.727508,7.263961,0.677267,0.062500


KeyboardInterrupt: 